# Phase 2 eval — Colab Free/T4 subset (16 models)Repository: Sudharsanselvaraj/Identifiable-Variance-Decomposition-of-Correlated-Errors-in-Foundation-ModelsRun cells IN ORDER. Free Colab sessions die; the loop resumes via git (rows are pushed after each model). If the session restarts: re-run cells 2-6, 7-9 (pilot gate), then 10 (production skips done models).PREREQUISITES (done once, outside Colab):1. Accept the 5 gated licenses: Llama-1, Mistral-Small-3, Mistral-Small-3.2, Devstral-2, Gemma-3n2. Create an HF read token (huggingface.co/settings/tokens)

In [ ]:
import platform, subprocess, sys, torchprint("python", sys.version.split()[0], "|", platform.platform())print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)print("torch", torch.__version__, "cuda", torch.cuda.is_available(),      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")# runbook expects Python 3.11 + lm-eval 0.4.12 (pinned below); python 3.12/3.13 usually fine

In [ ]:
%%bashcd /contentgit clone https://github.com/Sudharsanselvaraj/Identifiable-Variance-Decomposition-of-Correlated-Errors-in-Foundation-Models.git lineage-repocd lineage-repogit branch --show-currentecho "--- clean tree (must print nothing):"git status --porcelaingit log -1 --oneline

In [ ]:
%%bashcd /content/lineage-repo/srcpython3 -m lineage_era.phase2_eval --manifest | tail -1    # expect "47/47 manifest complete."grep -c load_in_4bit lineage_era/phase2_eval.py             # expect >= 1

In [ ]:
%%bashpython3 -m pip install -q lm-eval==0.4.12 accelerate bitsandbytespython3 -c "import torch, lm_eval; assert torch.cuda.is_available(); print('lm_eval', lm_eval.__version__, '| torch', torch.__version__)"

In [ ]:
%%bash# PASTE YOUR TOKEN HERE (placeholder only — never commit it)export HF_TOKEN="PASTE_YOUR_TOKEN_HERE"python3 -c "import os; from huggingface_hub import whoami; print('logged in as:', whoami()['name'])"

In [ ]:
import osos.environ["HF_HOME"] = "/content/hf_cache"os.makedirs("/content/hf_cache", exist_ok=True)print("HF_HOME:", os.environ["HF_HOME"])

In [ ]:
%%bash# PILOT 1/2 — Phi-2, 100 questions (real evaluator, pilot mode). Not research results.cd /content/lineage-repo/srcpython3 -m lineage_era.phase2_eval --model Phi-2 --limit 100 \  --device cuda:0 --attn sdpa --dtype float16 --quant 4bit

In [ ]:
# PILOT CLEANUP (ALL pilots) — drop any row that is not full MMLU (14042 samples)# and delete every pilot JSONL. Safe here: no production rows exist yet.import csv, os, globp = "../datasets/phase2_eval_results.csv"rows = list(csv.DictReader(open(p)))kept = [r for r in rows if int(r.get("samples") or 0) == 14042]removed = len(rows) - len(kept)assert removed == len(rows), "a full-MMLU row already exists — STOP and fix manually"open(p, "w").close() if removed == 0 else Nonewith open(p, "w", newline="") as f:    if kept:        w = csv.DictWriter(f, fieldnames=kept[0].keys()); w.writeheader(); w.writerows(kept)for s in glob.glob("../datasets/eval_samples/*.jsonl"):    os.remove(s)print(f"removed {removed} pilot row(s) + all pilot JSONL; CSV rows now: {len(kept)}")

In [ ]:
%%bash# PILOT 2/2 — fit/execution check for the 7 risky T4 models (12-24B).# Purpose is ONLY "does it load + run without OOM under production flags".# 50 questions each; expect 2-6 min of compute after the (large) weight download.cd /content/lineage-repo/srcexport HF_TOKEN="PASTE_YOUR_TOKEN_HERE"export HF_HOME=/content/hf_cachefor m in Phi-3 Phi-4-reasoning-vision-15B Gemma-4-12B \         Mistral-Small-3 Mistral-Small-3.1 Mistral-Small-3.2 Devstral-2; do  echo "========== pilot $m =========="  python3 -m lineage_era.phase2_eval --model "$m" --limit 50 \      --device cuda:0 --attn sdpa --dtype float16 --quant 4bit \    || { echo "PILOT FAILED $m — do NOT start production; diagnose first"; break; }  nvidia-smi --query-gpu=memory.used --format=csv,noheader  rm -rf /content/hf_cache/hub    # free disk before the next pilotdoneecho "PILOT GATE DONE"

In [ ]:
# PILOT CLEANUP (again) — remove the 7 pilot rows + their JSONL before productionimport csv, os, globp = "../datasets/phase2_eval_results.csv"rows = list(csv.DictReader(open(p)))kept = [r for r in rows if int(r.get("samples") or 0) == 14042]open(p, "w", newline="").close()with open(p, "w", newline="") as f:    if kept:        w = csv.DictWriter(f, fieldnames=kept[0].keys()); w.writeheader(); w.writerows(kept)for s in glob.glob("../datasets/eval_samples/*.jsonl"):    os.remove(s)print(f"removed {len(rows)-len(kept)} pilot rows; CSV rows now: {len(kept)}")assert len(kept) == 0, "unexpected pre-existing production rows"

In [ ]:
%%bash# PRODUCTION — 16 models, one at a time. After each: verify full MMLU (14042 samples),# commit + push, then delete only the model cache. Resume-safe: skips done models.cd /content/lineage-repo/srcexport HF_TOKEN="PASTE_YOUR_TOKEN_HERE"export HF_HOME=/content/hf_cachefor m in $(python3 -c "import csv;print(' '.join(r['full_name'] for r in csv.DictReader(open('../datasets/coverage/colab_t4_subset.csv'))))"); do  echo "========== $m =========="  python3 -m lineage_era.phase2_run_all --only "$m" --device cuda:0 \      --dtype float16 --attn sdpa --batch auto --quant 4bit || { echo "FAILED $m"; break; }  python3 -c "import csvrows=[r for r in csv.DictReader(open('../datasets/phase2_eval_results.csv')) if r['full_name']=='$m']assert len(rows)==1 and int(rows[0]['samples'])==14042, rowsassert rows[0]['fidelity']=='4bit'print('verified $m:', rows[0]['acc'], 'samples', rows[0]['samples'])" || break  cd /content/lineage-repo  git add datasets/phase2_eval_results.csv datasets/eval_samples/  git commit -m "eval: $m (T4, 4bit)"  git push  rm -rf /content/hf_cache/hub          # model cache only — repo artifacts untouched  cd /content/lineage-repo/srcdoneecho "LOOP DONE"

In [ ]:
%%bash# FINAL VERIFICATION — exactly 16 rows, 14042 samples each, A15 common item setcd /content/lineage-repo/srcpython3 -m lineage_era.analysis.eval_check \  --manifest ../datasets/coverage/colab_t4_subset.csv \  --csv ../datasets/phase2_eval_results.csv \  --samples-dir ../datasets/eval_samples

STOP here. Do NOT run the A100 models and do NOT run imputation/decomposition in Colab.If all 16 passed, hand off to the A100 (not Colab):    python3 -m lineage_era.phase2_run_all --subset ../datasets/coverage/a100_80gb_subset.csv \      --device cuda:0 --dtype bfloat16 --attn sdpa --quant 4bitThe A100 pulls the T4 rows and skips them automatically. Never run Colab and A100 concurrently (one shared CSV).